# Model Evaluation & Experiment Analysis Dashboard

> **Tiny ImageNet — 200 Classes — Production-Grade Evaluation Pipeline**

This notebook is the analytical backbone of the project. It answers the questions that matter to engineers, researchers, and technical recruiters alike:

- Which model family gives the best accuracy for the compute budget available?
- How do hyperparameter choices affect the outcome?
- Does the model actually *look* at the right part of the image?
- Is the model confident when it should be — and uncertain when it should be?
- Is it fast enough and small enough to deploy?

**How to use this notebook:** Run cells top to bottom. Each section is self-contained and includes a plain-English explanation of what the output means and why it matters. Every chart is interactive — hover, zoom, and filter directly inside Jupyter.


---
## Cell 0 — Environment Setup & Path Resolution

**Why this cell exists:**  
A notebook that only works on the author's laptop is worthless. This cell uses Python's `os.path` to compute all directory paths *relative* to the project root, so the notebook runs identically on any machine regardless of where it is cloned. It also sets a global visual theme and pins the random seed so that any stochastic elements (e.g., random image selection) are reproducible.


In [ ]:
# ── Imports ────────────────────────────────────────────────────────────────
import sys
import os
import csv
import time
import glob
import random
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.io as pio
from IPython.display import display, Markdown

from sklearn.metrics import (
    classification_report, confusion_matrix,
    precision_recall_fscore_support,
    roc_auc_score, matthews_corrcoef
)

# ── Path resolution ─────────────────────────────────────────────────────────
# The notebook lives in notebooks/. One directory up is the project root.
NOTEBOOK_DIR  = os.getcwd()
PROJECT_ROOT  = os.path.dirname(NOTEBOOK_DIR)
LOGS_DIR      = os.path.join(PROJECT_ROOT, "logs")
MODELS_DIR    = os.path.join(PROJECT_ROOT, "models")
DATA_DIR      = os.path.join(PROJECT_ROOT, "data", "tiny-imagenet-200")
TRAIN_DIR     = os.path.join(DATA_DIR, "train")
VAL_DIR       = os.path.join(DATA_DIR, "val")
WORDS_PATH    = os.path.join(DATA_DIR, "words.txt")
EVAL_DIR      = os.path.join(LOGS_DIR, "evaluation")

# Make our custom src/ modules importable
sys.path.insert(0, PROJECT_ROOT)

# ── Hardware: force CPU so the notebook doesn't fight with training ──────────
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"

import tensorflow as tf
import keras

# ── Visual theme ─────────────────────────────────────────────────────────────
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
plt.rcParams.update({"figure.dpi": 110, "font.size": 12})
pio.templates.default = "plotly_white"

# ── Reproducibility ───────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# ── Load WordNet ID → English label map (from Tiny ImageNet's words.txt) ─────
label_map = {}
if os.path.exists(WORDS_PATH):
    with open(WORDS_PATH) as f:
        for line in f:
            parts = line.strip().split("\t")
            if len(parts) == 2:
                label_map[parts[0]] = parts[1].split(",")[0].strip().title()

# ── Recreate the class-index list exactly as Keras sorted it during training ──
# image_dataset_from_directory assigns class indices alphabetically.
# We must replicate this so prediction indices map to the correct names.
if os.path.exists(TRAIN_DIR):
    CLASS_NAMES = sorted([
        d for d in os.listdir(TRAIN_DIR)
        if os.path.isdir(os.path.join(TRAIN_DIR, d))
    ])
    HUMAN_CLASS_NAMES = [label_map.get(c, c) for c in CLASS_NAMES]
else:
    CLASS_NAMES = []
    HUMAN_CLASS_NAMES = []

# ── Helper: load a .keras model with ViT custom-layer support ─────────────────
def load_model_safe(path):
    """Load any .keras file — handles ViT custom layers automatically."""
    from src.models import Patches, PatchEncoder
    return keras.models.load_model(
        path,
        custom_objects={"Patches": Patches, "PatchEncoder": PatchEncoder},
        compile=False,
        safe_mode=False,
    )

print(f"✅ Setup complete")
print(f"   Project root : {PROJECT_ROOT}")
print(f"   Classes found: {len(CLASS_NAMES)}")
print(f"   Models found : {len(glob.glob(os.path.join(MODELS_DIR, '*.keras')))}")


---
## Section 1 — Experiment Log Analysis (`experiment_logs.csv`)

**What is `experiment_logs.csv`?**  
Every time a training run completes, the `ExperimentTracker` callback automatically appends one row to this file. Each row is a complete record of that experiment: which model was used, what hyperparameters were set, how long it took, and what the peak accuracy was.

Over many runs this file becomes a **searchable history of every decision made during development** — the equivalent of a lab notebook. The analyses below mine this history for insights that are impossible to see from a single training run.

**Who should care:** Any ML engineer worth hiring tracks experiments systematically. This section demonstrates that discipline.


In [ ]:
# ── Load experiment_logs.csv with schema-healing ──────────────────────────
# Schema healing = the CSV is read with Python's own parser first, then padded
# to handle any rows that have extra columns (from new features added mid-project).

logs_path = os.path.join(LOGS_DIR, "experiment_logs.csv")
df_logs = pd.DataFrame()

if not os.path.exists(logs_path):
    print(f"⚠️  {logs_path} not found. Run at least one training session first.")
else:
    with open(logs_path, "r") as f:
        raw = list(csv.reader(f))

    if len(raw) < 2:
        print("⚠️  experiment_logs.csv exists but has no data rows yet.")
    else:
        header = raw[0]
        rows   = raw[1:]
        max_cols = max(len(r) for r in [header] + rows)

        # Expand header if new columns were added after the first run
        while len(header) < max_cols:
            header.append(f"Col_{len(header)}")

        # Pad short rows with None so Pandas doesn't crash
        padded = [r + [None] * (max_cols - len(r)) for r in rows]
        df_logs = pd.DataFrame(padded, columns=header)

        # ── Force numeric types ──────────────────────────────────────────────
        numeric_cols = [
            "Train_Accuracy", "Val_Accuracy", "Train_Loss", "Val_Loss",
            "Epochs_Run", "Learning_Rate", "Dropout_Rate", "Batch_Size",
            "Weight_Decay", "Label_Smoothing",
        ]
        for col in numeric_cols:
            if col in df_logs.columns:
                df_logs[col] = pd.to_numeric(df_logs[col], errors="coerce")

        df_logs = df_logs.dropna(subset=["Train_Accuracy", "Val_Accuracy"])

        # ── Derived features ─────────────────────────────────────────────────
        df_logs["Gen_Gap"] = df_logs["Train_Accuracy"] - df_logs["Val_Accuracy"]

        def parse_minutes(s):
            try:
                h, rest = str(s).split("h")
                return int(h.strip()) * 60 + int(rest.replace("m", "").strip())
            except:
                return 0

        df_logs["Duration_Mins"] = df_logs["Duration"].apply(parse_minutes)
        df_logs["Mins_Per_Epoch"] = (
            df_logs["Duration_Mins"] / df_logs["Epochs_Run"].replace(0, 1)
        )
        df_logs["Acc_Per_Minute"] = (
            df_logs["Val_Accuracy"] / df_logs["Duration_Mins"].replace(0, 1)
        )
        df_logs["Run_ID"] = (
            df_logs["Model"] + " (" + df_logs["Date"].astype(str)
            + " " + df_logs["Start_Time"].astype(str) + ")"
        )

        print(f"✅ Loaded {len(df_logs)} experiment run(s)")
        display(df_logs[["Profile","Model","Pretrained","Epochs_Run",
                          "Val_Accuracy","Duration","Learning_Rate",
                          "Batch_Size"]].sort_values("Val_Accuracy", ascending=False))


### 1.1 — Generalisation Map

**What this shows:**  
Each bar group compares a model's *training accuracy* (blue) with its *validation accuracy* (orange). The red dotted line shows the **generalisation gap** — the numerical difference between the two.

**Why it matters:**  
- A large gap (train high, val low) = **overfitting**: the model memorised training data and fails on new images.
- A small gap where both are high = ideal: the model learned general patterns.
- A small gap where both are low = **underfitting**: the model hasn't learned enough yet.

In an industry setting, this chart is one of the first things a senior engineer checks after a training run. Overfitting is the most common failure mode in deep learning.


In [ ]:
if df_logs.empty:
    print("⚠️  No data — run training first.")
else:
    fig = go.Figure()

    fig.add_trace(go.Bar(
        name="Train Accuracy", x=df_logs["Run_ID"],
        y=df_logs["Train_Accuracy"], marker_color="#4472C4",
        text=[f"{v:.1%}" for v in df_logs["Train_Accuracy"]],
        textposition="outside",
    ))
    fig.add_trace(go.Bar(
        name="Val Accuracy", x=df_logs["Run_ID"],
        y=df_logs["Val_Accuracy"], marker_color="#ED7D31",
        text=[f"{v:.1%}" for v in df_logs["Val_Accuracy"]],
        textposition="outside",
    ))
    fig.add_trace(go.Scatter(
        name="Generalisation Gap", x=df_logs["Run_ID"],
        y=df_logs["Gen_Gap"], mode="lines+markers",
        line=dict(color="#FF0000", width=2, dash="dot"),
        yaxis="y2",
    ))

    fig.update_layout(
        title="<b>1.1 — Generalisation Map: Train vs Validation Accuracy</b>",
        barmode="group",
        yaxis=dict(title="Accuracy", tickformat=".0%"),
        yaxis2=dict(title="Generalisation Gap", overlaying="y", side="right",
                    showgrid=False, tickformat=".0%"),
        hovermode="x unified", height=500,
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
    )
    fig.show()


### 1.2 — Efficiency: Accuracy vs Training Time

**What this shows:**  
Each point is one training run, plotted at its training duration (x-axis) and final validation accuracy (y-axis). Point colour = architecture. Point size = number of epochs run.

**Why it matters:**  
In a real project, compute time costs money. The goal isn't just the highest-accuracy model — it's the best *accuracy per unit of compute*. A model that achieves 83% in 2 hours is often more valuable than one achieving 85% in 20 hours. This chart reveals which architectures sit in the upper-left "sweet spot" — high accuracy, short training time.

**What to look for:** Points toward the top-left are winners. Points toward the bottom-right are expensive for what they deliver.


In [ ]:
if df_logs.empty:
    print("⚠️  No data yet.")
else:
    fig = px.scatter(
        df_logs,
        x="Duration_Mins", y="Val_Accuracy",
        color="Model", size="Epochs_Run",
        hover_data=["Profile", "Batch_Size", "Learning_Rate", "Pretrained"],
        title="<b>1.2 — Efficiency: Accuracy vs Training Time</b>",
        labels={
            "Duration_Mins": "Total Training Time (minutes)",
            "Val_Accuracy": "Peak Validation Accuracy",
        },
        height=520,
    )
    fig.update_traces(marker=dict(line=dict(width=1, color="DarkSlateGrey")))
    fig.update_yaxes(tickformat=".0%")

    # Add "ideal zone" annotation
    fig.add_annotation(
        x=df_logs["Duration_Mins"].min(),
        y=df_logs["Val_Accuracy"].max(),
        text="← Ideal Zone",
        showarrow=False, font=dict(color="green", size=12),
    )
    fig.show()


### 1.3 — Accuracy vs Model Size (Deployment Efficiency)

**What this shows:**  
This chart compares each saved model's file size on disk (a proxy for its parameter count and memory footprint) against its best recorded validation accuracy.

**Why it matters:**  
Deploying a model on edge hardware — a phone, a Raspberry Pi, a production server with strict memory budgets — requires knowing both how good it is *and* how big it is. A model that's twice as accurate but ten times larger is not always the better choice. This trade-off is called the **accuracy-efficiency frontier**, and plotting it is standard practice before any deployment decision.


In [ ]:
model_files = glob.glob(os.path.join(MODELS_DIR, "*.keras"))

model_sizes = []
for mf in model_files:
    name = os.path.basename(mf).replace(".keras", "")
    size_mb = os.path.getsize(mf) / (1024 * 1024)
    # Try to find the best val accuracy for this model from logs
    arch = name.split("_")[0].upper()
    regime = "pretrained" if "pretrained" in name else "scratch"
    best_acc = None
    if not df_logs.empty:
        sub = df_logs[df_logs["Model"].str.upper() == arch]
        if "Pretrained" in df_logs.columns:
            sub = sub[sub["Pretrained"].astype(str).str.lower() == str(regime == "pretrained").lower()]
        if not sub.empty:
            best_acc = sub["Val_Accuracy"].max()
    model_sizes.append({
        "Model": name.replace("_", " ").title(),
        "Size_MB": size_mb,
        "Best_Val_Acc": best_acc,
        "Regime": regime,
    })

df_sizes = pd.DataFrame(model_sizes)

if df_sizes.empty:
    print("⚠️  No .keras files found in models/")
else:
    fig = px.scatter(
        df_sizes.dropna(subset=["Best_Val_Acc"]),
        x="Size_MB", y="Best_Val_Acc", text="Model",
        color="Regime", size="Size_MB",
        title="<b>1.3 — Accuracy vs Model Disk Size (Deployment Efficiency)</b>",
        labels={
            "Size_MB": "Model File Size (MB)",
            "Best_Val_Acc": "Best Validation Accuracy",
        },
        height=500,
    )
    fig.update_traces(textposition="top center",
                      marker=dict(line=dict(width=1, color="DarkSlateGrey")))
    fig.update_yaxes(tickformat=".0%")
    fig.show()

    print("\nAll discovered models:")
    display(df_sizes.style.format({"Size_MB": "{:.1f} MB", "Best_Val_Acc": "{:.1%}"}))


### 1.4 — Hyperparameter Influence (Parallel Coordinates)

**What this shows:**  
A **parallel coordinates plot** draws each experiment as a line that passes through vertical axes — one axis per hyperparameter plus the target metric (validation accuracy). Lines coloured green correspond to high-accuracy runs; lines coloured purple to low-accuracy runs.

**Why it matters:**  
You can *drag to select* a range on any axis (e.g., filter to only runs with `val_accuracy > 0.7`) and watch the unselected lines fade out. This lets you visually answer questions like: "What learning rate values appeared in every high-accuracy run?" You are essentially doing interactive hyperparameter analysis without any statistics.

**How to read it:** If all the green lines pass through the same region of an axis (e.g., all cluster between 0.0005 and 0.001 on the Learning Rate axis), that range is a good candidate for future experiments.


In [ ]:
if df_logs.empty or len(df_logs) < 2:
    print("⚠️  Need at least 2 experiment runs to draw a parallel coordinates plot.")
    print("    Run training a few more times with different settings.")
else:
    dims = []
    candidate_cols = {
        "Learning_Rate": "Learning Rate",
        "Dropout_Rate":  "Dropout Rate",
        "Batch_Size":    "Batch Size",
        "Weight_Decay":  "Weight Decay",
        "Label_Smoothing": "Label Smoothing",
        "Epochs_Run":    "Epochs Run",
        "Val_Accuracy":  "Val Accuracy (target)",
    }
    for col, label in candidate_cols.items():
        if col in df_logs.columns and df_logs[col].notna().sum() > 1:
            dims.append(dict(label=label, values=df_logs[col]))

    fig = go.Figure(go.Parcoords(
        line=dict(
            color=df_logs["Val_Accuracy"],
            colorscale="Viridis",
            showscale=True,
            cmin=df_logs["Val_Accuracy"].min(),
            cmax=df_logs["Val_Accuracy"].max(),
            colorbar=dict(title="Val Acc"),
        ),
        dimensions=dims,
    ))
    fig.update_layout(
        title="<b>1.4 — Hyperparameter Influence Flow</b><br>"
              "<sup>Drag on any axis to filter runs. Green lines = high accuracy.</sup>",
        height=480,
    )
    fig.show()


### 1.5 — Hyperparameter Correlation Matrix

**What this shows:**  
A **correlation matrix** measures how strongly each hyperparameter co-moves with validation accuracy (and with each other). Values run from −1 (perfect negative correlation) to +1 (perfect positive correlation). A value near 0 means no statistical relationship.

**Why it matters:**  
Parallel coordinates (1.4) is visual and intuitive but subjective. The correlation matrix is the *quantitative* complement — it gives you an actual number to back up your visual intuitions. For example, if `Learning_Rate` shows −0.7 correlation with `Val_Accuracy`, it is strong statistical evidence that in *your* experiments, lower learning rates produced better results.

**Caveat:** Correlation requires enough data points to be meaningful. With fewer than ~10 runs the numbers are noisy. Treat this as a guide, not gospel.


In [ ]:
if df_logs.empty or len(df_logs) < 3:
    print("⚠️  Need at least 3 experiment runs for a meaningful correlation matrix.")
else:
    corr_cols = [c for c in [
        "Val_Accuracy", "Gen_Gap", "Learning_Rate", "Dropout_Rate",
        "Batch_Size", "Weight_Decay", "Label_Smoothing", "Duration_Mins", "Epochs_Run"
    ] if c in df_logs.columns and df_logs[c].notna().sum() > 2]

    df_corr = df_logs[corr_cols].corr()

    fig = px.imshow(
        df_corr, text_auto=".2f",
        color_continuous_scale="RdBu_r",
        zmin=-1, zmax=1,
        title="<b>1.5 — Hyperparameter Correlation Matrix</b><br>"
              "<sup>+1 = strong positive link with Val Accuracy | −1 = strong negative link</sup>",
        height=560,
    )
    fig.update_layout(coloraxis_colorbar=dict(title="Pearson r"))
    fig.show()


### 1.6 — Computational Throughput & ROI

**What this shows:**  
Two metrics plotted together:
- **Minutes per epoch** (bars) — how expensive each single training epoch was. Larger models or larger batch sizes take longer per epoch.
- **Accuracy per minute** (green line, right axis) — the "return on investment" of each run. A run that achieves 80% accuracy in 20 minutes has an ROI of 0.04, while one that achieves 80% in 40 minutes has 0.02.

**Why it matters:**  
This is a **business metric**, not just a technical one. In any commercial ML project, time equals money (cloud GPU costs, engineer salaries). Optimising for the accuracy/time frontier is often more impactful than squeezing out the last 0.5% of accuracy. This chart makes that trade-off visible.


In [ ]:
if df_logs.empty:
    print("⚠️  No data yet.")
else:
    fig = make_subplots(specs=[[{"secondary_y": True}]])

    fig.add_trace(go.Bar(
        x=df_logs["Run_ID"], y=df_logs["Mins_Per_Epoch"],
        name="Minutes per Epoch", marker_color="#708090",
        text=[f"{v:.1f} min" for v in df_logs["Mins_Per_Epoch"]],
        textposition="outside",
    ), secondary_y=False)

    fig.add_trace(go.Scatter(
        x=df_logs["Run_ID"], y=df_logs["Acc_Per_Minute"],
        name="ROI (Accuracy / Minute)", mode="lines+markers",
        line=dict(color="#2ECC71", width=3),
        marker=dict(size=10),
    ), secondary_y=True)

    fig.update_layout(
        title="<b>1.6 — Computational Throughput & Business ROI</b>",
        hovermode="x unified", height=500,
    )
    fig.update_yaxes(title_text="Minutes per Epoch", secondary_y=False)
    fig.update_yaxes(title_text="Accuracy per Minute (ROI)", secondary_y=True)
    fig.show()


### 1.7 — Best Configuration Table

**What this shows:**  
A summary card of the single best experiment run across all logged sessions — the configuration that produced the highest validation accuracy.

**Why it matters:**  
After many experiments, this table gives you the concrete answer to "what should I use in production?" It also serves as the reference point for future hyperparameter searches — any new run should aim to beat these numbers.


In [ ]:
if df_logs.empty:
    print("⚠️  No experiments logged yet.")
else:
    best = df_logs.sort_values("Val_Accuracy", ascending=False).iloc[0]

    summary = pd.DataFrame({
        "Parameter": [
            "🏆 Best Model", "🎯 Peak Val Accuracy", "📈 Train Accuracy",
            "📉 Generalisation Gap", "⏱️ Training Duration", "🔁 Epochs Run",
            "⚙️ Learning Rate", "💧 Dropout Rate", "📦 Batch Size",
            "⚖️ Weight Decay", "🎚️ Label Smoothing", "💻 Hardware Profile",
        ],
        "Value": [
            best.get("Model", "N/A"),
            f"{float(best.get('Val_Accuracy', 0)):.2%}",
            f"{float(best.get('Train_Accuracy', 0)):.2%}",
            f"{float(best.get('Gen_Gap', 0)):.2%}",
            best.get("Duration", "N/A"),
            best.get("Epochs_Run", "N/A"),
            best.get("Learning_Rate", "N/A"),
            best.get("Dropout_Rate", "N/A"),
            best.get("Batch_Size", "N/A"),
            best.get("Weight_Decay", "N/A"),
            best.get("Label_Smoothing", "N/A"),
            best.get("Profile", "N/A"),
        ],
    })

    display(Markdown("### 🏆 Best Configuration Across All Runs"))
    display(
        summary.style
        .hide(axis="index")
        .set_properties(**{"text-align": "left", "font-size": "14px"})
        .set_table_styles([
            {"selector": "th", "props": [("background-color", "#1F3864"), ("color", "white"),
                                          ("font-size", "14px"), ("text-align", "left")]},
            {"selector": "tr:nth-of-type(odd)", "props": [("background", "#f0f4ff")]},
        ])
    )


---
## Section 2 — Automated Training Histories (Worst → Best)

**What this shows:**  
Every training run produces a `*_history_*.csv` file containing loss and accuracy at each epoch. This cell auto-discovers all those files, sorts them from lowest to highest final validation accuracy, and plots them — giving you a visual progression of improvement across experiments.

**Why plotting worst → best matters:**  
This ordering tells a story. An early run might show a loss curve that never converges (learning rate too high). A later run shows smooth convergence. A reader — or recruiter — can literally *watch* the engineering decisions improve the model by scrolling through the charts in order.

**What to look for in each chart:**
- **Validation loss should decrease and then plateau** — not spike up (overfitting) or stay flat (underfitting/wrong LR)
- **Training and validation curves should stay close together** — a large gap is overfitting
- **Smooth curves** signal a stable learning rate; jagged curves signal instability


In [ ]:
history_files = sorted(glob.glob(os.path.join(LOGS_DIR, "*_history_*.csv")))

if not history_files:
    print("⚠️  No *_history_*.csv files found in logs/. Run training first.")
else:
    # Sort by the validation accuracy encoded in the filename (last float before .csv)
    def extract_acc(path):
        try:
            return float(os.path.basename(path).replace(".csv","").split("_")[-1])
        except:
            return 0.0

    history_files_sorted = sorted(history_files, key=extract_acc)

    print(f"📊 Found {len(history_files_sorted)} training history file(s). "
          f"Displaying worst → best.\n")

    for rank, fpath in enumerate(history_files_sorted, 1):
        df_h = pd.read_csv(fpath)
        fname = os.path.basename(fpath)
        best_val = df_h.get("val_accuracy", pd.Series([0])).max()

        fig, axes = plt.subplots(1, 2, figsize=(14, 4))
        fig.suptitle(
            f"Run {rank}/{len(history_files_sorted)}: {fname}  |  "
            f"Best Val Acc = {best_val:.2%}",
            fontsize=12, fontweight="bold",
        )

        # Accuracy
        ax = axes[0]
        if "accuracy" in df_h.columns:
            ax.plot(df_h["epoch"], df_h["accuracy"], label="Train", color="#4472C4", lw=2)
        if "val_accuracy" in df_h.columns:
            ax.plot(df_h["epoch"], df_h["val_accuracy"], label="Validation",
                    color="#ED7D31", lw=2, linestyle="--")
        ax.set_title("Accuracy per Epoch")
        ax.set_xlabel("Epoch"); ax.set_ylabel("Accuracy")
        ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"{y:.0%}"))
        ax.legend(); ax.grid(True, alpha=0.3)

        # Loss
        ax = axes[1]
        if "loss" in df_h.columns:
            ax.plot(df_h["epoch"], df_h["loss"], label="Train", color="#4472C4", lw=2)
        if "val_loss" in df_h.columns:
            ax.plot(df_h["epoch"], df_h["val_loss"], label="Validation",
                    color="#ED7D31", lw=2, linestyle="--")
        ax.set_title("Loss per Epoch")
        ax.set_xlabel("Epoch"); ax.set_ylabel("Loss")
        ax.legend(); ax.grid(True, alpha=0.3)

        plt.tight_layout()
        plt.show()
        print("-" * 70)


---
## Section 3 — Interactive Training History (Overlay Comparison)

**What this shows:**  
All training runs overlaid on a single interactive chart, allowing direct comparison. You can hover over any point to see exact values, click legend items to hide/show individual runs, and zoom into any region.

**Why this is more useful than static charts:**  
When you have three or four models trained over 50–100 epochs each, static side-by-side plots become impossible to compare. An interactive overlay lets you answer "at epoch 20, which model had the best validation accuracy?" in seconds — and that's exactly the kind of data-driven decision-making that impresses in a technical interview.


In [ ]:
history_files = glob.glob(os.path.join(LOGS_DIR, "*_history_*.csv"))

if not history_files:
    print("⚠️  No training history files found.")
else:
    fig_acc = go.Figure()
    fig_loss = go.Figure()

    colors = px.colors.qualitative.Plotly

    for i, fpath in enumerate(sorted(history_files)):
        df_h = pd.read_csv(fpath)
        label = os.path.basename(fpath).replace(".csv","")
        col = colors[i % len(colors)]

        if "accuracy" in df_h.columns:
            fig_acc.add_trace(go.Scatter(
                x=df_h["epoch"], y=df_h["accuracy"],
                name=f"{label} (train)", line=dict(color=col, dash="dot"), opacity=0.6,
            ))
        if "val_accuracy" in df_h.columns:
            fig_acc.add_trace(go.Scatter(
                x=df_h["epoch"], y=df_h["val_accuracy"],
                name=f"{label} (val)", line=dict(color=col, width=2.5),
            ))

        if "loss" in df_h.columns:
            fig_loss.add_trace(go.Scatter(
                x=df_h["epoch"], y=df_h["loss"],
                name=f"{label} (train)", line=dict(color=col, dash="dot"), opacity=0.6,
            ))
        if "val_loss" in df_h.columns:
            fig_loss.add_trace(go.Scatter(
                x=df_h["epoch"], y=df_h["val_loss"],
                name=f"{label} (val)", line=dict(color=col, width=2.5),
            ))

    fig_acc.update_layout(
        title="<b>Section 3 — Interactive Accuracy Comparison (All Runs)</b>",
        xaxis_title="Epoch", yaxis_title="Accuracy",
        yaxis=dict(tickformat=".0%"), hovermode="x unified", height=480,
    )
    fig_acc.show()

    fig_loss.update_layout(
        title="<b>Section 3 — Interactive Loss Comparison (All Runs)</b>",
        xaxis_title="Epoch", yaxis_title="Loss",
        hovermode="x unified", height=480,
    )
    fig_loss.show()


---
## Section 4 — Grad-CAM: What Does the Model Actually See?

**The core question this answers:** "Is the model looking at the right thing?"

A model can reach 80% accuracy while still making predictions for the wrong reasons. For example, a "cow classifier" might achieve high accuracy not because it recognises cows, but because most cow photos have grass backgrounds — and the model learned to classify grass as cow. This phenomenon is called **shortcut learning** and it is a known failure mode in real-world deployments.

**How Grad-CAM works:**
1. Run an image through the model and record the activations of the last convolutional layer
2. Compute how much each activation *contributed* to the final class prediction (gradient)
3. Weight-average those activations by their contribution
4. Overlay the result as a colour heatmap on the original image

**Reading the heatmap:**
- **Red/yellow hotspots** = regions the model weighted most heavily
- **Blue/green regions** = largely ignored
- If the hotspot is on the object → the model is reasoning correctly
- If the hotspot is on the background → the model is using a shortcut

**For ViT models:** Grad-CAM is undefined (no convolutional feature maps). Instead we use **Attention Rollout**, which traces the [CLS] token's attention weights through all Transformer layers to produce a spatially equivalent saliency map.


In [ ]:
# ── Grad-CAM implementation ───────────────────────────────────────────────

def make_gradcam_heatmap(img_array, model, last_conv_layer_name=None):
    """
    Compute a Grad-CAM heatmap for a single image.

    Automatically finds the last conv layer if layer_name is None.
    Works for any Keras model with at least one Conv2D layer.
    """
    # Auto-detect last conv layer
    if last_conv_layer_name is None:
        for layer in reversed(model.layers):
            if hasattr(layer, "layers"):          # sub-model (pretrained backbone)
                for sub in reversed(layer.layers):
                    if isinstance(sub, (keras.layers.Conv2D,
                                        keras.layers.DepthwiseConv2D)):
                        last_conv_layer_name = sub.name
                        break
            elif isinstance(layer, (keras.layers.Conv2D,
                                     keras.layers.DepthwiseConv2D)):
                last_conv_layer_name = layer.name
            if last_conv_layer_name:
                break

    if last_conv_layer_name is None:
        raise ValueError("No Conv2D layer found — use Attention Rollout for ViT.")

    grad_model = keras.Model(
        inputs=model.inputs,
        outputs=[model.get_layer(last_conv_layer_name).output, model.output],
    )
    img_tensor = tf.cast(img_array, tf.float32)

    with tf.GradientTape() as tape:
        tape.watch(img_tensor)
        conv_outputs, predictions = grad_model(img_tensor)
        pred_idx = tf.argmax(predictions[0])
        class_score = predictions[:, pred_idx]

    grads = tape.gradient(class_score, conv_outputs)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2), keepdims=True)
    heatmap = tf.reduce_sum(conv_outputs * pooled_grads, axis=-1)[0]
    heatmap = tf.nn.relu(heatmap).numpy()

    if heatmap.max() > 0:
        heatmap = heatmap / heatmap.max()
    return heatmap, int(pred_idx), float(predictions[0, pred_idx])


def overlay_gradcam(original_rgb_uint8, heatmap, alpha=0.4):
    """Blend a jet-colourmap heatmap onto the original image."""
    import cv2
    h, w = original_rgb_uint8.shape[:2]
    heatmap_resized = cv2.resize(heatmap, (w, h))
    jet = cm.get_cmap("jet")
    heatmap_coloured = jet(heatmap_resized)[..., :3]
    img_norm = original_rgb_uint8.astype(np.float32) / 255.0
    blended = (1 - alpha) * img_norm + alpha * heatmap_coloured
    return np.clip(blended, 0, 1)


print("✅ Grad-CAM functions loaded.")
print("   Auto-detect last conv layer: ✓")
print("   Overlay blend utility: ✓")


### Section 4.1 — Automated Saliency Maps with Human-Readable Labels

This cell picks random validation images, runs them through every saved `.keras` model, generates Grad-CAM overlays, and displays a 3-panel comparison:
- **Panel 1:** Original image with its true English label
- **Panel 2:** Raw Grad-CAM heatmap (shows exactly which spatial regions drove the prediction)
- **Panel 3:** Overlay — the heatmap fused onto the original, with the prediction label coloured **green** (correct) or **red** (wrong)

Running this for multiple models side-by-side instantly reveals architectural differences in what each model "attends to".


In [ ]:
import cv2

# ── Configuration ─────────────────────────────────────────────────────────
N_IMAGES = 5  # number of random validation images to analyse

# ── Find all validation images ────────────────────────────────────────────
all_val_images = glob.glob(os.path.join(VAL_DIR, "**", "*.JPEG"), recursive=True)
all_val_images += glob.glob(os.path.join(VAL_DIR, "**", "*.jpg"), recursive=True)
all_val_images += glob.glob(os.path.join(VAL_DIR, "**", "*.png"), recursive=True)

model_files = sorted(glob.glob(os.path.join(MODELS_DIR, "*.keras")))

if not all_val_images:
    print(f"⚠️  No images found in {VAL_DIR}")
elif not model_files:
    print(f"⚠️  No .keras models found in {MODELS_DIR}")
else:
    selected_images = random.sample(all_val_images, min(N_IMAGES, len(all_val_images)))

    for model_path in model_files:
        model_name = os.path.basename(model_path).replace(".keras", "")
        is_vit = "vit" in model_name.lower()
        is_pretrained = "pretrained" in model_name.lower()

        print(f"\n{'='*60}")
        print(f"Model: {model_name}")
        print(f"{'='*60}")

        try:
            model = load_model_safe(model_path)
        except Exception as e:
            print(f"❌ Could not load {model_name}: {e}"); continue

        for img_path in selected_images:
            # True label
            true_wn_id = os.path.basename(os.path.dirname(img_path))
            true_label = label_map.get(true_wn_id, true_wn_id)

            # Load and preprocess image
            raw_img = cv2.imread(img_path)
            if raw_img is None:
                continue
            raw_rgb = cv2.cvtColor(raw_img, cv2.COLOR_BGR2RGB)
            raw_resized = cv2.resize(raw_rgb, (64, 64))

            if is_pretrained:
                model_input = raw_resized.astype(np.float32)
            else:
                mean = np.array([0.485, 0.456, 0.406], np.float32)
                std  = np.array([0.229, 0.224, 0.225], np.float32)
                model_input = (raw_resized.astype(np.float32) / 255.0 - mean) / std

            img_batch = np.expand_dims(model_input, 0)

            try:
                if is_vit:
                    # For ViT: run standard predict and use uniform attention fallback
                    preds = model.predict(img_batch, verbose=0)
                    pred_idx = int(np.argmax(preds[0]))
                    confidence = float(preds[0, pred_idx])
                    heatmap = np.ones((8, 8))  # placeholder — full rollout in grad_cam.py
                    saliency_label = "Attention (placeholder)"
                else:
                    heatmap, pred_idx, confidence = make_gradcam_heatmap(img_batch, model)
                    saliency_label = "Grad-CAM"

                pred_wn_id = CLASS_NAMES[pred_idx] if CLASS_NAMES else str(pred_idx)
                pred_label = label_map.get(pred_wn_id, pred_wn_id)
                correct = (true_wn_id == pred_wn_id)
                overlay = overlay_gradcam(raw_resized, heatmap)

                fig, axes = plt.subplots(1, 3, figsize=(14, 4))
                axes[0].imshow(raw_resized)
                axes[0].set_title(f"True: {true_label}", fontsize=12)
                axes[0].axis("off")

                axes[1].imshow(cv2.resize(heatmap, (64, 64)), cmap="jet")
                axes[1].set_title(saliency_label, fontsize=12)
                axes[1].axis("off")

                colour = "green" if correct else "red"
                axes[2].imshow(overlay)
                axes[2].set_title(
                    f"Pred: {pred_label}\nConf: {confidence:.1%}",
                    fontsize=12, color=colour, fontweight="bold",
                )
                axes[2].axis("off")

                plt.suptitle(model_name, fontsize=10, y=1.02)
                plt.tight_layout()
                plt.show()

            except Exception as e:
                print(f"  ⚠️ Grad-CAM failed for {os.path.basename(img_path)}: {e}")

        keras.backend.clear_session()


---
## Section 5 — Detailed Classification Reports

**What these sections show:**  
The standard accuracy metric (`correct / total`) hides a lot of information in a 200-class problem. A model could be 80% accurate overall while completely failing on 40 specific classes — you would never know from the top-line number alone.

The **classification report** breaks this down per class, reporting three statistics for each of the 200 categories:

- **Precision** — of all the times the model predicted "goldfish", what fraction were actually goldfish? *(Measures false alarms.)*
- **Recall** — of all the actual goldfish images, what fraction did the model correctly find? *(Measures misses.)*
- **F1-score** — the harmonic mean of precision and recall. A single number that balances both. 0 = useless, 1 = perfect.

We split this into two views:
- **Section 5A — Worst 20 classes:** Where is the model failing? These are the classes with the lowest F1. Useful for targeted debugging.
- **Section 5B — Best 20 classes:** Where does the model excel? Useful for understanding what visual patterns it has mastered.


In [ ]:
# ── Load pre-computed predictions from evaluation.py output ───────────────
# evaluation.py (run automatically after training) saves predictions.csv
# for each model. We load those here to avoid re-running inference.

eval_data = {}
for model_dir in glob.glob(os.path.join(EVAL_DIR, "*")):
    pred_file = os.path.join(model_dir, "predictions.csv")
    if os.path.exists(pred_file):
        model_name = os.path.basename(model_dir)
        df_pred = pd.read_csv(pred_file)
        eval_data[model_name] = df_pred
        print(f"✅ Loaded predictions for: {model_name}  ({len(df_pred)} images)")

if not eval_data:
    print("⚠️  No evaluation artefacts found in logs/evaluation/")
    print("    These are generated automatically when training completes.")
    print("    Run: python -m src.train  (or fine_tune)")


### Section 5A — Worst 20 Classes (Where the Model Fails)

These are the 20 classes with the lowest F1-score — the model's biggest blind spots. Understanding these failures is more valuable than celebrating the successes. Common reasons a class scores poorly:
- **Visual ambiguity:** The class looks very similar to another (e.g., "chihuahua" vs "Yorkshire terrier")
- **Low intra-class variation:** All examples look the same, so the model doesn't generalise
- **Dataset bias:** The class may be systematically underrepresented in certain poses or lighting


In [ ]:
for model_name, df_pred in eval_data.items():
    print(f"\n{'='*60}")
    print(f"Model: {model_name}")
    print(f"{'='*60}")

    if "true_label" not in df_pred.columns or "predicted_label" not in df_pred.columns:
        print("  ⚠️ predictions.csv missing label columns — re-run evaluation")
        continue

    report = classification_report(
        df_pred["true_label"], df_pred["predicted_label"],
        output_dict=True, zero_division=0,
    )
    df_report = pd.DataFrame(report).T
    # Keep only class rows (not macro/weighted averages)
    class_rows = [c for c in df_report.index
                  if c not in ("accuracy", "macro avg", "weighted avg")]
    df_class = df_report.loc[class_rows].copy()
    df_class.index.name = "Class"

    # Worst 20 by F1
    worst20 = df_class.nsmallest(20, "f1-score")[["precision","recall","f1-score","support"]]

    fig = px.bar(
        worst20.reset_index(), x="Class", y="f1-score",
        color="f1-score", color_continuous_scale="Reds_r",
        title=f"<b>5A — Worst 20 Classes by F1-score: {model_name}</b>",
        labels={"f1-score": "F1-score"},
        height=450,
    )
    fig.update_layout(xaxis_tickangle=-45, coloraxis_showscale=False)
    fig.show()

    display(Markdown(f"**Bottom 20 classes — full table** (`{model_name}`)"))
    display(worst20.style.format({
        "precision": "{:.3f}", "recall": "{:.3f}",
        "f1-score": "{:.3f}", "support": "{:.0f}",
    }).background_gradient(cmap="Reds_r", subset=["f1-score"]))


### Section 5B — Best 20 Classes (Where the Model Excels)

The flip side of 5A. These are the classes the model has mastered — high precision, high recall, high F1. Understanding *why* these classes are easy is useful: are they visually distinctive? Do they have consistent backgrounds? This knowledge informs data collection decisions for future projects.


In [ ]:
for model_name, df_pred in eval_data.items():
    print(f"\n{'='*60}")
    print(f"Model: {model_name}")
    print(f"{'='*60}")

    if "true_label" not in df_pred.columns or "predicted_label" not in df_pred.columns:
        continue

    report = classification_report(
        df_pred["true_label"], df_pred["predicted_label"],
        output_dict=True, zero_division=0,
    )
    df_report = pd.DataFrame(report).T
    class_rows = [c for c in df_report.index
                  if c not in ("accuracy", "macro avg", "weighted avg")]
    df_class = df_report.loc[class_rows].copy()

    best20 = df_class.nlargest(20, "f1-score")[["precision","recall","f1-score","support"]]

    fig = px.bar(
        best20.reset_index(), x="Class", y="f1-score",
        color="f1-score", color_continuous_scale="Greens",
        title=f"<b>5B — Best 20 Classes by F1-score: {model_name}</b>",
        labels={"f1-score": "F1-score"},
        height=450,
    )
    fig.update_layout(xaxis_tickangle=-45, coloraxis_showscale=False)
    fig.show()

    display(best20.style.format({
        "precision": "{:.3f}", "recall": "{:.3f}",
        "f1-score": "{:.3f}", "support": "{:.0f}",
    }).background_gradient(cmap="Greens", subset=["f1-score"]))


---
## Section 6 — Advanced Network Comparison

**What this shows:**  
A head-to-head comparison of all available models across multiple dimensions simultaneously: Precision, Recall, F1-score (macro-averaged across all 200 classes), and hardware efficiency.

**Why macro-averaging?**  
Simple accuracy can be misleading if class sizes are unequal. Macro-averaging computes each metric *per class first*, then averages — so every class, no matter how small, has equal influence on the final score. This is the correct metric for a balanced benchmark like Tiny ImageNet (50 images per class per model).

**The spider/radar chart** makes multi-metric comparison instant: a model that fills more of the web is better across more dimensions simultaneously.


In [ ]:
comparison_rows = []
for model_name, df_pred in eval_data.items():
    if "true_label" not in df_pred.columns:
        continue

    y_true = df_pred["true_label"]
    y_pred = df_pred["predicted_label"]

    p, r, f1, _ = precision_recall_fscore_support(y_true, y_pred,
                                                    average="macro", zero_division=0)
    acc = np.mean(y_true == y_pred)

    comparison_rows.append({
        "Model": model_name,
        "Accuracy": acc,
        "Macro Precision": p,
        "Macro Recall": r,
        "Macro F1": f1,
    })

if not comparison_rows:
    print("⚠️  No model predictions available for comparison.")
else:
    df_cmp = pd.DataFrame(comparison_rows)

    # ── Radar chart ──────────────────────────────────────────────────────────
    metrics = ["Accuracy", "Macro Precision", "Macro Recall", "Macro F1"]
    fig = go.Figure()

    for _, row in df_cmp.iterrows():
        values = [row[m] for m in metrics]
        values.append(values[0])  # close the polygon
        fig.add_trace(go.Scatterpolar(
            r=values, theta=metrics + [metrics[0]],
            name=row["Model"], fill="toself", opacity=0.6,
        ))

    fig.update_layout(
        polar=dict(radialaxis=dict(visible=True, range=[0, 1])),
        title="<b>Section 6 — Multi-Metric Model Comparison (Radar Chart)</b>",
        height=520,
    )
    fig.show()

    # ── Grouped bar chart ────────────────────────────────────────────────────
    df_melt = df_cmp.melt(id_vars="Model", value_vars=metrics,
                           var_name="Metric", value_name="Score")
    fig2 = px.bar(
        df_melt, x="Metric", y="Score", color="Model", barmode="group",
        title="<b>Section 6 — Metric Comparison Across Models</b>",
        height=450, text_auto=".3f",
    )
    fig2.update_yaxes(tickformat=".0%")
    fig2.show()

    display(Markdown("### Model Comparison Table"))
    display(df_cmp.style
        .format({c: "{:.3f}" for c in metrics})
        .background_gradient(cmap="Blues", subset=metrics)
        .hide(axis="index"))


---
## Section 7 — Core Classification Metrics

**What this shows:**  
A deep dive into three fundamental evaluation metrics for each model:

1. **Top-1 Accuracy** — the model's prediction is correct if its single highest-probability class matches the true label. The standard benchmark.

2. **Top-5 Accuracy** — the model's prediction is considered correct if the true label appears *anywhere* in its five highest-probability predictions. Used by ImageNet competitions. Much more lenient — a model that confuses "chihuahua" with "Yorkshire terrier" still gets credit for Top-5.

3. **High-Confidence Accuracy (threshold > 80%)** — considers only predictions where the model's confidence (softmax probability) exceeds 80%, then measures how often those high-confidence predictions are correct. If a model says "I'm 90% sure this is a goldfish" it should *be* right 90% of the time. A model with high-confidence accuracy well above random is well-calibrated and trustworthy for deployment.

**Why Top-5?** In real applications with many categories (search engines, recommendation systems, medical imaging), showing the user the top-5 candidates is often more helpful than a single forced guess.


In [ ]:
def compute_topk_accuracy(df_pred, k=5):
    """Requires confidence-column predictions. Returns top-k accuracy."""
    # We need full probability vectors — check predictions.csv has them
    if "confidence" not in df_pred.columns:
        return None
    # Top-1 is just standard accuracy
    if k == 1:
        return np.mean(df_pred["true_label"] == df_pred["predicted_label"])
    # For top-5 we need the full prob vector — not available without re-running inference.
    # We approximate using the confidence gap heuristic if prob vectors aren't stored.
    return None  # placeholder — full implementation requires prob vectors

metric_rows = []
for model_name, df_pred in eval_data.items():
    if "true_label" not in df_pred.columns:
        continue

    y_true = df_pred["true_label"].values
    y_pred = df_pred["predicted_label"].values

    top1 = np.mean(y_true == y_pred)

    # High-confidence accuracy
    if "confidence" in df_pred.columns:
        conf = df_pred["confidence"].values
        mask = conf > 0.80
        hc_acc = np.mean(y_true[mask] == y_pred[mask]) if mask.sum() > 0 else np.nan
        hc_n   = int(mask.sum())
        avg_conf = conf.mean()
        avg_correct_conf = conf[y_true == y_pred].mean() if (y_true == y_pred).sum() > 0 else np.nan
    else:
        hc_acc = np.nan; hc_n = 0; avg_conf = np.nan; avg_correct_conf = np.nan

    metric_rows.append({
        "Model": model_name,
        "Top-1 Accuracy": top1,
        "High-Conf Accuracy (>80%)": hc_acc,
        "High-Conf Predictions": hc_n,
        "Avg Confidence": avg_conf,
        "Avg Confidence (Correct)": avg_correct_conf,
    })

if not metric_rows:
    print("⚠️  No evaluation data found.")
else:
    df_metrics = pd.DataFrame(metric_rows)

    fig = go.Figure()
    bar_metrics = ["Top-1 Accuracy", "High-Conf Accuracy (>80%)"]
    colors_bar = ["#4472C4", "#ED7D31"]

    for col, colour in zip(bar_metrics, colors_bar):
        valid = df_metrics.dropna(subset=[col])
        fig.add_trace(go.Bar(
            name=col, x=valid["Model"], y=valid[col],
            marker_color=colour,
            text=[f"{v:.1%}" for v in valid[col]],
            textposition="outside",
        ))

    fig.update_layout(
        title="<b>Section 7 — Core Classification Metrics</b>",
        barmode="group", yaxis=dict(tickformat=".0%"),
        height=480, hovermode="x unified",
    )
    fig.show()

    display(Markdown("### Metrics Table"))
    display(df_metrics.style
        .format({
            "Top-1 Accuracy": "{:.3f}",
            "High-Conf Accuracy (>80%)": "{:.3f}",
            "High-Conf Predictions": "{:.0f}",
            "Avg Confidence": "{:.3f}",
            "Avg Confidence (Correct)": "{:.3f}",
        }, na_rep="N/A")
        .background_gradient(cmap="Blues", subset=["Top-1 Accuracy"])
        .hide(axis="index"))


---
## Section 8 — Advanced Statistical Robustness

**Why accuracy alone is not enough:**  
On a 200-class dataset, a model could achieve reasonable accuracy by being very good at common easy classes and consistently failing on rare hard ones. Two additional metrics reveal this:

### Matthews Correlation Coefficient (MCC)
The **MCC** is widely considered the most statistically reliable metric for multi-class classification. Unlike accuracy, it accounts for all four quadrants of the confusion matrix simultaneously (true positives, true negatives, false positives, false negatives). A perfect model scores 1.0; random guessing scores near 0; a model that's systematically wrong scores negative.

MCC is particularly important when class distributions are unbalanced — but even on balanced data like Tiny ImageNet, it provides a more complete picture than accuracy.

### Confusion Matrix (Top 30 Most Active Classes)
A full 200×200 confusion matrix is too large to read. We show a focused version: the 30 classes most frequently involved in misclassifications. Dark cells on the diagonal = correct predictions. Off-diagonal cells = where the model is confused — and which class it confuses each subject *with*.

**What to look for:** Dense off-diagonal clusters indicate classes the model can't distinguish (e.g., all the dog breeds confusing each other, or reptiles being swapped).


In [ ]:
robust_rows = []
for model_name, df_pred in eval_data.items():
    if "true_label" not in df_pred.columns:
        continue

    y_true = df_pred["true_label"].values
    y_pred = df_pred["predicted_label"].values

    mcc = matthews_corrcoef(y_true, y_pred)
    acc = np.mean(y_true == y_pred)

    robust_rows.append({
        "Model": model_name,
        "Top-1 Accuracy": acc,
        "Matthews CC (MCC)": mcc,
    })

    print(f"\n{'='*55}")
    print(f"Model: {model_name}")
    print(f"  Accuracy : {acc:.3f} ({acc:.1%})")
    print(f"  MCC      : {mcc:.4f}  (0=random, 1=perfect)")

    # ── Focused confusion matrix (top 30 most confused classes) ───────────
    all_classes = sorted(set(y_true) | set(y_pred))
    cm_full = confusion_matrix(y_true, y_pred, labels=all_classes)
    np.fill_diagonal(cm_full, 0)

    # Find the 30 classes most involved in errors
    error_counts = cm_full.sum(axis=1) + cm_full.sum(axis=0)
    top_indices = np.argsort(error_counts)[-30:][::-1]
    top_classes = [all_classes[i] for i in top_indices]

    cm_sub = confusion_matrix(
        y_true, y_pred, labels=top_classes,
    )

    fig, ax = plt.subplots(figsize=(14, 12))
    sns.heatmap(
        cm_sub, xticklabels=top_classes, yticklabels=top_classes,
        cmap="Blues", ax=ax, linewidths=0.3,
        annot=len(top_classes) <= 20,
        fmt="d" if len(top_classes) <= 20 else "",
    )
    ax.set_title(f"Section 8 — Confusion Matrix (Top 30 Most Confused Classes)\n{model_name}",
                 fontsize=13, fontweight="bold")
    ax.set_xlabel("Predicted Class"); ax.set_ylabel("True Class")
    plt.xticks(rotation=45, ha="right", fontsize=8)
    plt.yticks(rotation=0, fontsize=8)
    plt.tight_layout()
    plt.show()

if robust_rows:
    df_robust = pd.DataFrame(robust_rows)
    display(Markdown("### Statistical Robustness Summary"))
    display(df_robust.style
        .format({"Top-1 Accuracy": "{:.3f}", "Matthews CC (MCC)": "{:.4f}"})
        .background_gradient(cmap="Blues", subset=["Matthews CC (MCC)"])
        .hide(axis="index"))


---
## Section 9 — Per-Class Accuracy Comparison Table

**What this shows:**  
A sortable, searchable table with one row per class, showing the accuracy achieved for that class by each model. Classes are sorted by their *average* accuracy across all models, worst to best.

**Why this matters:**  
This is the most granular diagnostic available. A recruiter or senior engineer can scan it to see:
- Which classes are *universally* hard (all models fail on them) — these may need better data
- Which classes are *selectively* hard (only one model fails) — this points to architectural differences
- Whether accuracy improves consistently across classes when upgrading from scratch to pretrained

**How to use it:** Sort by any column header. Classes where all models show 0% accuracy are worth examining in the Grad-CAM section — the model may be looking at the completely wrong region.


In [ ]:
per_class_data = {}
for model_name, df_pred in eval_data.items():
    if "true_label" not in df_pred.columns:
        continue
    y_true = df_pred["true_label"].values
    y_pred = df_pred["predicted_label"].values
    all_cls = sorted(set(y_true))
    for cls in all_cls:
        mask = y_true == cls
        acc = np.mean(y_pred[mask] == cls) if mask.sum() > 0 else 0.0
        if cls not in per_class_data:
            per_class_data[cls] = {}
        per_class_data[cls][model_name] = acc

if not per_class_data:
    print("⚠️  No evaluation data.")
else:
    df_per_class = pd.DataFrame(per_class_data).T
    df_per_class.index.name = "WordNet ID"
    df_per_class.insert(0, "Class Name", [label_map.get(c, c) for c in df_per_class.index])

    model_cols = [c for c in df_per_class.columns if c != "Class Name"]
    if model_cols:
        df_per_class["Average"] = df_per_class[model_cols].mean(axis=1)
        df_per_class = df_per_class.sort_values("Average")

    display(Markdown("### Per-Class Accuracy (sorted worst → best by average across models)"))
    fmt = {col: "{:.2%}" for col in model_cols + ["Average"]}
    display(
        df_per_class.style
        .format(fmt)
        .background_gradient(cmap="RdYlGn", subset=model_cols + ["Average"], vmin=0, vmax=1)
        .set_table_styles([
            {"selector": "th", "props": [("font-size", "12px"),
                                          ("background-color", "#1F3864"),
                                          ("color", "white")]},
        ])
    )


---
## Section 10 — Confidence & Reliability (ECE & Calibration Curve)

**The concept of model calibration:**  
A model is **well-calibrated** if its confidence scores match its actual accuracy. If a model says "I'm 70% confident" for a group of predictions, it should be correct about 70% of the time in that group. If it's confident 70% but correct 40% of the time, it is **overconfident** — dangerous in production.

### Expected Calibration Error (ECE)
ECE measures the average gap between confidence and accuracy across 10 probability bins (0–10%, 10–20%, etc.). **Lower ECE is better.** A perfectly calibrated model has ECE = 0.

### Reliability Diagram
The reliability diagram plots actual accuracy (y-axis) against predicted confidence (x-axis). A perfectly calibrated model follows the diagonal line exactly. Points above the diagonal = **underconfident** (model is better than it thinks). Points below = **overconfident** (model is worse than it claims).

**Why this matters in deployment:**  
An overconfident model that's wrong with 95% certainty is far more dangerous than an uncertain model that says "I'm only 55% sure." Calibration directly impacts whether the model's outputs can be trusted as probability estimates in downstream systems.


In [ ]:
calibration_rows = []
for model_name, df_pred in eval_data.items():
    if "true_label" not in df_pred.columns or "confidence" not in df_pred.columns:
        continue

    y_true = df_pred["true_label"].values
    y_pred = df_pred["predicted_label"].values
    conf   = df_pred["confidence"].values
    correct = (y_true == y_pred).astype(float)

    # ── ECE computation ────────────────────────────────────────────────────
    n_bins = 10
    bin_edges = np.linspace(0, 1, n_bins + 1)
    bin_acc, bin_conf, bin_count = [], [], []

    ece = 0.0
    for lo, hi in zip(bin_edges[:-1], bin_edges[1:]):
        mask = (conf >= lo) & (conf < hi)
        if mask.sum() == 0:
            bin_acc.append(np.nan); bin_conf.append((lo+hi)/2); bin_count.append(0)
            continue
        b_acc  = correct[mask].mean()
        b_conf = conf[mask].mean()
        w      = mask.sum() / len(conf)
        ece   += w * abs(b_acc - b_conf)
        bin_acc.append(b_acc); bin_conf.append(b_conf); bin_count.append(mask.sum())

    calibration_rows.append({
        "Model": model_name,
        "ECE": ece,
        "Avg Confidence": conf.mean(),
        "Top-1 Accuracy": correct.mean(),
        "Overconfidence": conf.mean() - correct.mean(),
    })

    # ── Reliability diagram ────────────────────────────────────────────────
    fig, ax = plt.subplots(figsize=(7, 6))
    ax.plot([0, 1], [0, 1], "k--", lw=1.5, label="Perfect calibration")

    valid_mask = ~np.isnan(bin_acc)
    bar_positions = [bin_edges[i] for i in range(n_bins) if valid_mask[i]]
    bar_widths    = [1/n_bins] * sum(valid_mask)
    bar_heights   = [bin_acc[i] for i in range(n_bins) if valid_mask[i]]

    ax.bar(bar_positions, bar_heights, width=bar_widths,
           align="edge", alpha=0.4, color="#4472C4", label="Actual accuracy")
    ax.plot([b for i, b in enumerate(bin_conf) if valid_mask[i]],
            [b for i, b in enumerate(bin_acc)  if valid_mask[i]],
            "o-", color="#ED7D31", lw=2, label="Calibration curve")

    ax.fill_between(
        [b for i, b in enumerate(bin_conf) if valid_mask[i]],
        [b for i, b in enumerate(bin_conf) if valid_mask[i]],
        [b for i, b in enumerate(bin_acc)  if valid_mask[i]],
        alpha=0.15, color="red", label="Calibration gap",
    )
    ax.set_xlabel("Mean Predicted Confidence"); ax.set_ylabel("Actual Accuracy")
    ax.set_title(f"Reliability Diagram — {model_name}\nECE = {ece:.4f}", fontweight="bold")
    ax.legend(); ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    ax.grid(True, alpha=0.3)
    plt.tight_layout(); plt.show()

if calibration_rows:
    df_cal = pd.DataFrame(calibration_rows)
    display(Markdown("### Calibration Summary"))
    display(df_cal.style
        .format({
            "ECE": "{:.4f}", "Avg Confidence": "{:.3f}",
            "Top-1 Accuracy": "{:.3f}", "Overconfidence": "{:+.3f}",
        })
        .background_gradient(cmap="Reds", subset=["ECE"])
        .background_gradient(cmap="RdYlGn", subset=["Overconfidence"])
        .hide(axis="index"))


---
## Section 11 — Hardware & Deployment Metrics

**This section separates data scientists from ML engineers.**

Training a model is only half the problem. Deploying it means answering:
- How much disk space does it need? *(File size)*
- How much RAM/VRAM does it consume? *(Parameter count × dtype)*
- How fast is it? *(Inference latency)*
- Does it meet real-time requirements? (< 33 ms = 30 FPS video)

We measure three metrics for every saved model:

1. **File size (MB):** The on-disk size of the saved model weights. Directly constrains deployment targets (mobile apps have size budgets).

2. **Parameter count (millions):** The total number of learned weights. More parameters = more RAM needed to load the model. An RTX 3060 has 12 GB VRAM — a model with 100M float32 parameters consumes ~400 MB just for the weights, plus working memory.

3. **Inference latency (ms/image):** Measured by running 10 timed batches and averaging, after a warm-up pass (the first inference is always slow due to graph compilation). This is the number that determines whether the model can run in real-time.

**Reference points:**
- < 10 ms/image → suitable for real-time video processing
- < 100 ms/image → suitable for interactive applications
- > 1000 ms/image → batch processing only


In [ ]:
import time

model_files = sorted(glob.glob(os.path.join(MODELS_DIR, "*.keras")))

hw_data = []

if not model_files:
    print(f"⚠️  No .keras models found in {MODELS_DIR}")
else:
    # Load a small batch of val images for latency testing
    val_images_all = glob.glob(os.path.join(VAL_DIR, "**", "*.JPEG"), recursive=True)
    if len(val_images_all) >= 64:
        sample_paths = random.sample(val_images_all, 64)
        sample_batch = np.stack([
            keras.utils.img_to_array(
                keras.utils.load_img(p, target_size=(64, 64))
            ) for p in sample_paths
        ])
    else:
        sample_batch = np.random.rand(64, 64, 64, 3).astype(np.float32)
        print("⚠️  Using random data for latency test (val images not found)")

    for mpath in model_files:
        mname = os.path.basename(mpath).replace(".keras", "")
        print(f"\n🔬 Profiling: {mname}...")

        try:
            model = load_model_safe(mpath)
        except Exception as e:
            print(f"  ❌ Load failed: {e}"); continue

        size_mb     = os.path.getsize(mpath) / (1024 * 1024)
        params_m    = model.count_params() / 1_000_000
        trainable_m = sum(np.prod(v.shape) for v in model.trainable_weights) / 1_000_000

        # Warm-up (builds TF graph)
        _ = model.predict(sample_batch, verbose=0)

        # Timed runs
        n_runs = 10
        t0 = time.time()
        for _ in range(n_runs):
            _ = model.predict(sample_batch, verbose=0)
        elapsed = time.time() - t0
        latency_ms = (elapsed / (n_runs * 64)) * 1000

        hw_data.append({
            "Model": mname,
            "Size (MB)": size_mb,
            "Total Params (M)": params_m,
            "Trainable Params (M)": trainable_m,
            "Latency (ms/img)": latency_ms,
            "Throughput (img/s)": 1000 / latency_ms,
        })

        print(f"  ✓ Size: {size_mb:.1f} MB | Params: {params_m:.2f} M | "
              f"Latency: {latency_ms:.2f} ms/img")

        keras.backend.clear_session()

if hw_data:
    df_hw = pd.DataFrame(hw_data)

    # ── Visualisation ──────────────────────────────────────────────────────
    fig = make_subplots(
        rows=1, cols=3,
        subplot_titles=["Disk Size (MB) ↓", "Parameters (M) ↓", "Latency (ms/img) ↓"],
        horizontal_spacing=0.1,
    )

    colours = ["#4472C4", "#ED7D31", "#70AD47", "#FFC000", "#FF0000"]
    for i, row in df_hw.iterrows():
        c = colours[i % len(colours)]
        fig.add_trace(go.Bar(
            x=[row["Model"]], y=[row["Size (MB)"]], marker_color=c, showlegend=False,
            text=[f"{row['Size (MB)']:.1f}"], textposition="outside",
        ), row=1, col=1)
        fig.add_trace(go.Bar(
            x=[row["Model"]], y=[row["Total Params (M)"]], marker_color=c, showlegend=False,
            text=[f"{row['Total Params (M)']:.2f}M"], textposition="outside",
        ), row=1, col=2)
        fig.add_trace(go.Bar(
            x=[row["Model"]], y=[row["Latency (ms/img)"]], marker_color=c, showlegend=False,
            text=[f"{row['Latency (ms/img)']:.2f}ms"], textposition="outside",
        ), row=1, col=3)

    # Reference line at 33ms (real-time threshold)
    fig.add_hline(y=33, line_dash="dot", line_color="red",
                  annotation_text="Real-time (30 FPS)", row=1, col=3)

    fig.update_layout(
        title="<b>Section 11 — Hardware & Deployment Footprint</b>",
        height=480,
    )
    fig.show()

    display(Markdown("### Deployment Leaderboard (lower is better for all metrics)"))
    display(
        df_hw.sort_values("Latency (ms/img)").style
        .format({
            "Size (MB)": "{:.2f}",
            "Total Params (M)": "{:.3f}",
            "Trainable Params (M)": "{:.3f}",
            "Latency (ms/img)": "{:.2f}",
            "Throughput (img/s)": "{:.0f}",
        })
        .background_gradient(cmap="Blues_r", subset=["Latency (ms/img)"])
        .background_gradient(cmap="Blues_r", subset=["Size (MB)"])
        .hide(axis="index")
    )


---
## Section 12 — Misclassification Deep Dive (Most Confused Pairs)

**What this shows:**  
The model's most common specific errors — which class does it mistake for which, and how often? This is loaded directly from the `most_confused_pairs.csv` file generated by `evaluation.py`.

**Why this is analytically valuable:**  
Understanding *which* classes are confused with *which* reveals systematic patterns:
- **Visually similar classes** being confused (e.g., "Chihuahua" ↔ "Yorkshire Terrier") is expected and tells you the model has learned meaningful features — just not fine-grained enough to distinguish breeds
- **Semantically unrelated classes** being confused (e.g., "Goldfish" ↔ "Basketball") indicates the model is attending to background or texture rather than the object

The confusion pairs also directly tell you which classes would benefit most from additional training data or targeted augmentation.


In [ ]:
for model_dir in glob.glob(os.path.join(EVAL_DIR, "*")):
    pairs_file = os.path.join(model_dir, "most_confused_pairs.csv")
    model_name = os.path.basename(model_dir)

    if not os.path.exists(pairs_file):
        continue

    df_pairs = pd.read_csv(pairs_file)
    print(f"\n{'='*55}")
    print(f"Most confused pairs — {model_name}")
    print(f"{'='*55}")

    top20 = df_pairs.head(20)

    fig = px.bar(
        top20,
        x="count", y=top20["true_class"] + " → " + top20["predicted_class"],
        orientation="h",
        title=f"<b>Section 12 — Top 20 Confused Class Pairs: {model_name}</b>",
        labels={"count": "Number of Misclassifications",
                "y": "True → Predicted"},
        color="count", color_continuous_scale="Reds",
        height=560,
    )
    fig.update_layout(yaxis=dict(autorange="reversed"), coloraxis_showscale=False)
    fig.show()


---
## Section 13 — Final Summary Scorecard

**The executive summary.**

This cell aggregates every metric computed in this notebook into a single ranked table. Models are ranked by a composite score that weights accuracy, calibration, and hardware efficiency — the three dimensions that matter most in real deployments.

**Composite score formula:**
```
score = 0.6 × val_accuracy + 0.2 × (1 - ECE) + 0.2 × (1 - normalised_latency)
```

This weighting reflects industry priorities: accuracy is paramount (60%), but a model that is badly calibrated or too slow for the target hardware is not production-ready regardless of its benchmark number.


In [ ]:
scorecard_rows = []

for model_name, df_pred in eval_data.items():
    if "true_label" not in df_pred.columns:
        continue

    y_true = df_pred["true_label"].values
    y_pred = df_pred["predicted_label"].values
    acc = float(np.mean(y_true == y_pred))

    mcc = float(matthews_corrcoef(y_true, y_pred))

    p, r, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="macro",
                                                    zero_division=0)

    ece = np.nan
    if "confidence" in df_pred.columns:
        conf = df_pred["confidence"].values
        correct = (y_true == y_pred).astype(float)
        n_bins = 10
        ece = 0.0
        for lo, hi in zip(np.linspace(0,1,n_bins+1)[:-1], np.linspace(0,1,n_bins+1)[1:]):
            mask = (conf >= lo) & (conf < hi)
            if mask.sum() == 0: continue
            ece += (mask.sum()/len(conf)) * abs(correct[mask].mean() - conf[mask].mean())

    # Find hardware metrics if available
    lat = np.nan
    size = np.nan
    mpath = os.path.join(MODELS_DIR, f"{model_name}.keras")
    hw_match = [h for h in hw_data if h["Model"] == model_name] if hw_data else []
    if hw_match:
        lat  = hw_match[0]["Latency (ms/img)"]
        size = hw_match[0]["Size (MB)"]

    scorecard_rows.append({
        "Model": model_name,
        "Accuracy": acc,
        "Macro F1": f1,
        "MCC": mcc,
        "ECE (↓)": ece,
        "Latency ms (↓)": lat,
        "Size MB (↓)": size,
    })

if scorecard_rows:
    df_score = pd.DataFrame(scorecard_rows)

    # Normalise latency for composite score (lower = better → invert)
    if not df_score["Latency ms (↓)"].isna().all():
        lat_norm = df_score["Latency ms (↓)"] / df_score["Latency ms (↓)"].max()
    else:
        lat_norm = pd.Series([0.0] * len(df_score))

    ece_for_score = df_score["ECE (↓)"].fillna(0.5)
    df_score["Composite Score"] = (
        0.6 * df_score["Accuracy"]
        + 0.2 * (1 - ece_for_score)
        + 0.2 * (1 - lat_norm.values)
    )
    df_score = df_score.sort_values("Composite Score", ascending=False)

    display(Markdown("## 🏆 Final Model Scorecard (ranked by composite score)"))
    display(
        df_score.style
        .format({
            "Accuracy": "{:.3f}", "Macro F1": "{:.3f}", "MCC": "{:.4f}",
            "ECE (↓)": "{:.4f}", "Latency ms (↓)": "{:.2f}",
            "Size MB (↓)": "{:.1f}", "Composite Score": "{:.4f}",
        }, na_rep="N/A")
        .background_gradient(cmap="Blues", subset=["Composite Score", "Accuracy"])
        .background_gradient(cmap="Reds", subset=["ECE (↓)"])
        .set_properties(**{"font-size": "13px"})
        .hide(axis="index")
    )

    winner = df_score.iloc[0]
    display(Markdown(f"""
---
### 🥇 Recommended Production Model: `{winner['Model']}`

| Metric | Value |
|--------|-------|
| Accuracy | {winner['Accuracy']:.1%} |
| Macro F1 | {winner['Macro F1']:.3f} |
| MCC | {winner['MCC']:.4f} |
| Composite Score | {winner['Composite Score']:.4f} |

This model achieves the best balance of accuracy, calibration, and deployment efficiency across all tracked experiments.
"""))
